In [ ]:

from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

# Load environment variables
load_dotenv()
client = OpenAI()


# Step 1: Create your system prompt

system_prompt = """
You are J.A.R.V.I.S., Tony Stark's highly intelligent AI assistant.

You are an expert in Marvel Comics, superheroes,
villains, character histories and the MCU.

Your task is to research any Marvel character and produce
an entertaining, accurate and beautifully formatted report.

Use the following Markdown structure:

# 🦸 Character Name
Include the real name, alias and introduction.

## 📖 Origin Story
Explain the character's background and transformation.

## ⚡ Powers and Abilities
List their abilities with explanations.

## 🛡️ Weaknesses
Describe vulnerabilities and limitations.

## 🤝 Allies and Enemies
List their important allies, teams and enemies.

## 🎬 Marvel Comics vs MCU
Compare comic and movie versions where applicable.

## 🌟 Top 5 Interesting Facts
Give five surprising or entertaining facts.

## 🏆 J.A.R.V.I.S. Final Assessment
Provide an entertaining character assessment.

FORMATTING RULES:
- Use Markdown headings, tables and bullet points.
- Highlight important information in bold.
- Include suitable emojis.
- Keep paragraphs concise and entertaining.
- Prefer official Marvel sources.
- Clearly distinguish comic continuity from MCU continuity.
- Never invent unsupported facts.
- Include clickable source links supporting the claims.
- If information cannot be verified, explain this.
"""


# Step 2: Research and summarize any Marvel character

def summarize(character_name):

    character_name = character_name.strip()

    if not character_name:
        raise ValueError("Please enter a Marvel character name.")

    user_prompt = f"""
    Research the Marvel character: {character_name}

    Search for information using official Marvel websites first.
    Use other reputable sources if necessary.

    Identify the correct character and avoid confusing
    different people using the same superhero identity.

    Generate a detailed Marvel Character Intelligence Report
    using the requested Markdown structure.

    Include sources for your findings.
    """

    response = client.responses.create(
        model="gpt-4.1-mini",
        tools=[
            {
                "type": "web_search",
                "search_context_size": "medium"
            }
        ],
        tool_choice="required",
        input=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )

    # Extract source URLs from the research
    sources = {}

    for item in response.output:
        if item.type != "message":
            continue

        for part in item.content:
            for annotation in getattr(part, "annotations", []):
                if annotation.type == "url_citation":
                    sources[annotation.url] = annotation.title

    return response.output_text, sources


# Step 3: Display the character report

def display_summary(character_name):

    print(f"🔎 J.A.R.V.I.S. is researching {character_name}...\n")

    summary, sources = summarize(character_name)

    if not summary:
        print("No character information was found.")
        return

    # Display the report using Markdown
    display(Markdown(summary))

    # Display research sources
    if sources:
        display(Markdown("---\n### 🔗 Research Sources"))

        for url, title in sources.items():
            display(Markdown(f"- [{title or url}]({url})"))


# Step 4: Ask the user for a Marvel character

character_name = input("🦸 Enter a Marvel character's name: ").strip()

if character_name:
    display_summary(character_name)
else:
    print("Please enter a character name.")
